# Drug Development Landscape
## Part 1: Data Collection from the ClinicalTrials.gov API

This notebook collects clinical trial data from ClinicalTrials.gov API v2.

The dataset will be used to build a relational SQLite database for the analysis of global drug development trends.

The main objectives are:

- Retrieve interventional clinical trials.
- Focus on drug interventions.
- Extract information about sponsors, conditions, interventions, phases, and locations.
- Prepare raw data for database construction.

## Import Libraries
Import the libraries required for API requests, data manipulation, and workflow management.

In [1]:
import requests
import pandas as pd
import json
from tqdm import tqdm
import time

## API Request Functions
A function was created to retrieve clinical trial records from the ClinicalTrials.gov API. Pagination was implemented to allow the extraction of multiple study records.

In [5]:
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.term": "AREA[StudyType]INTERVENTIONAL AND AREA[InterventionType]DRUG",
    "pageSize": 100,
}

In [6]:
def download_trials(max_studies=5000):
    
    studies = []
    next_page_token = None
    
    while len(studies) < max_studies:
        
        params = {
            "query.term": "AREA[StudyType]INTERVENTIONAL AND AREA[InterventionType]DRUG",
            "pageSize": 100
        }
        
        if next_page_token:
            params["pageToken"] = next_page_token
        
        response = requests.get(BASE_URL, params=params)
        
        if response.status_code != 200:
            print("Error:", response.status_code)
            break
        
        data = response.json()
        
        studies.extend(data["studies"])
        
        print(f"Downloaded {len(studies)} studies")
        
        next_page_token = data.get("nextPageToken")
        
        if not next_page_token:
            break
            
        time.sleep(0.2)
    
    return studies[:max_studies]

## Download Clinical Trial Data
The API request returned 5,000 clinical trial records that will be processed and transformed into structured tables.

In [7]:
raw_trials = download_trials(max_studies=5000)

Downloaded 100 studies
Downloaded 200 studies
Downloaded 300 studies
Downloaded 400 studies
Downloaded 500 studies
Downloaded 600 studies
Downloaded 700 studies
Downloaded 800 studies
Downloaded 900 studies
Downloaded 1000 studies
Downloaded 1100 studies
Downloaded 1200 studies
Downloaded 1300 studies
Downloaded 1400 studies
Downloaded 1500 studies
Downloaded 1600 studies
Downloaded 1700 studies
Downloaded 1800 studies
Downloaded 1900 studies
Downloaded 2000 studies
Downloaded 2100 studies
Downloaded 2200 studies
Downloaded 2300 studies
Downloaded 2400 studies
Downloaded 2500 studies
Downloaded 2600 studies
Downloaded 2700 studies
Downloaded 2800 studies
Downloaded 2900 studies
Downloaded 3000 studies
Downloaded 3100 studies
Downloaded 3200 studies
Downloaded 3300 studies
Downloaded 3400 studies
Downloaded 3500 studies
Downloaded 3600 studies
Downloaded 3700 studies
Downloaded 3800 studies
Downloaded 3900 studies
Downloaded 4000 studies
Downloaded 4100 studies
Downloaded 4200 studies
D

In [8]:
with open("../data/raw_trials.json", "w") as f:
    json.dump(raw_trials, f)

## Explore JSON Structure
Each study record is returned as a nested JSON object containing different modules, including study identification, sponsors, interventions, conditions, and locations.

In [9]:
raw_trials[0].keys()

dict_keys(['protocolSection', 'derivedSection', 'hasResults'])

In [10]:
raw_trials[0]

{'protocolSection': {'identificationModule': {'nctId': 'NCT01042717',
   'orgStudyIdInfo': {'id': 'GCO # 09-0824'},
   'organization': {'fullName': 'Shi, Patricia, M.D.', 'class': 'INDIV'},
   'briefTitle': 'Study of the Best Timing for Plerixafor in Autologous Hematopoietic Stem Cell Collection',
   'officialTitle': 'Mobilization Kinetics of Plerixafor and G-CSF in Patients With NHL and MM Undergoing Autologous Peripheral Blood Progenitor Cell Collection'},
  'statusModule': {'statusVerifiedDate': '2011-09',
   'overallStatus': 'UNKNOWN',
   'lastKnownStatus': 'RECRUITING',
   'expandedAccessInfo': {'hasExpandedAccess': False},
   'startDateStruct': {'date': '2010-02'},
   'primaryCompletionDateStruct': {'date': '2011-12', 'type': 'ESTIMATED'},
   'completionDateStruct': {'date': '2011-12', 'type': 'ESTIMATED'},
   'studyFirstSubmitDate': '2010-01-05',
   'studyFirstSubmitQcDate': '2010-01-05',
   'studyFirstPostDateStruct': {'date': '2010-01-06', 'type': 'ESTIMATED'},
   'lastUpdateS

## Extract Study Information
The main study-level information was extracted, including study ID, title, recruitment status, clinical phase, and enrollment.

In [22]:
def extract_studies(trials):
    
    studies = []
    
    for trial in trials:
        
        protocol = trial["protocolSection"]
        
        study = {
            "study_id": protocol["identificationModule"].get("nctId"),
            "title": protocol["identificationModule"].get("briefTitle"),
            "status": protocol["statusModule"].get("overallStatus"),
            "phase": protocol["designModule"].get("phases", [None])[0],
            "enrollment": protocol["designModule"]
                .get("enrollmentInfo", {})
                .get("count")
        }
        
        studies.append(study)
    
    return pd.DataFrame(studies)

In [23]:
studies_df = extract_studies(raw_trials)

In [24]:
studies_df.head()

,study_id,title,status,phase,enrollment
0,NCT01042717,Study of the Best Timing for Plerixafor in Aut...,UNKNOWN,NA,10.0
1,NCT04941040,Opioid Free VS Opioid Anesthesia for Craniotomies,COMPLETED,PHASE1,60.0
2,NCT05883540,Lysergic Acid Diethylamide (LSD) in Palliative...,RECRUITING,PHASE2,60.0
3,NCT00171717,Conversion From Tacrolimus to Cyclosporine Mic...,COMPLETED,PHASE4,39.0
4,NCT00022217,Cisplatin-Epinephrine Injectable Gel Plus Pacl...,UNKNOWN,PHASE2,NaN


In [25]:
studies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   study_id    5000 non-null   str    
 1   title       5000 non-null   str    
 2   status      5000 non-null   str    
 3   phase       5000 non-null   str    
 4   enrollment  4949 non-null   float64
dtypes: float64(1), str(4)
memory usage: 195.4 KB


## Extract Sponsor Information
Sponsor information was extracted separately because each study can have multiple sponsors.

In [26]:
def extract_sponsors(trials):
    
    sponsors = []
    
    for trial in trials:
        
        protocol = trial["protocolSection"]
        
        study_id = protocol["identificationModule"].get("nctId")
        
        sponsor_module = protocol.get(
            "sponsorCollaboratorsModule",
            {}
        )
        
        # Lead sponsor
        lead = sponsor_module.get("leadSponsor")
        
        if lead:
            sponsors.append({
                "study_id": study_id,
                "sponsor_name": lead.get("name"),
                "sponsor_type": lead.get("class")
            })
        
        # Collaborators
        collaborators = sponsor_module.get(
            "collaborators",
            []
        )
        
        for collaborator in collaborators:
            
            sponsors.append({
                "study_id": study_id,
                "sponsor_name": collaborator.get("name"),
                "sponsor_type": collaborator.get("class")
            })
    
    return pd.DataFrame(sponsors)

In [27]:
sponsors_df = extract_sponsors(raw_trials)

In [28]:
sponsors_df.head()

,study_id,sponsor_name,sponsor_type
0,NCT01042717,"Shi, Patricia, M.D.",INDIV
1,NCT01042717,"Genzyme, a Sanofi Company",INDUSTRY
2,NCT04941040,Kasr El Aini Hospital,OTHER
3,NCT05883540,"University Hospital, Basel, Switzerland",OTHER
4,NCT05883540,"University Hospital, Zürich",OTHER


In [29]:
sponsors_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7590 entries, 0 to 7589
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   study_id      7590 non-null   str  
 1   sponsor_name  7590 non-null   str  
 2   sponsor_type  7590 non-null   str  
dtypes: str(3)
memory usage: 178.0 KB


In [50]:
sponsors_df["sponsor_name"].isna().sum()

np.int64(0)

## Extract Intervention Information
Interventions were extracted because a single clinical trial may evaluate multiple drugs or other intervention types.

In [30]:
def extract_interventions(trials):
    
    interventions = []
    
    for trial in trials:
        
        protocol = trial["protocolSection"]
        
        study_id = protocol["identificationModule"].get("nctId")
        
        intervention_module = protocol.get(
            "armsInterventionsModule",
            {}
        )
        
        study_interventions = intervention_module.get(
            "interventions",
            []
        )
        
        for intervention in study_interventions:
            
            interventions.append({
                "study_id": study_id,
                "intervention_name": intervention.get("name"),
                "intervention_type": intervention.get("type")
            })
    
    return pd.DataFrame(interventions)

In [31]:
interventions_df = extract_interventions(raw_trials)

In [32]:
interventions_df.head()

,study_id,intervention_name,intervention_type
0,NCT01042717,Plerixafor,DRUG
1,NCT04941040,Opioid free anesthetics,DRUG
2,NCT04941040,Opioid Anesthetics,DRUG
3,NCT05883540,Lysergic Acid Diethylamide Tartrate,DRUG
4,NCT05883540,Lysergic Acid Diethylamide Tartrate,DRUG


In [33]:
interventions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11304 entries, 0 to 11303
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   study_id           11304 non-null  str  
 1   intervention_name  11304 non-null  str  
 2   intervention_type  11304 non-null  str  
dtypes: str(3)
memory usage: 265.1 KB


In [48]:
interventions_df = interventions_df.drop_duplicates()

In [51]:
interventions_df["intervention_name"].isna().sum()

np.int64(0)

## Extract Condition Information
Disease conditions associated with each study were extracted to enable later analysis of therapeutic areas.

In [34]:
def extract_conditions(trials):
    
    conditions = []
    
    for trial in trials:
        
        protocol = trial["protocolSection"]
        
        study_id = protocol["identificationModule"].get("nctId")
        
        condition_module = protocol.get(
            "conditionsModule",
            {}
        )
        
        study_conditions = condition_module.get(
            "conditions",
            []
        )
        
        for condition in study_conditions:
            
            conditions.append({
                "study_id": study_id,
                "condition": condition
            })
    
    return pd.DataFrame(conditions)

In [35]:
conditions_df = extract_conditions(raw_trials)

In [36]:
conditions_df.head()

,study_id,condition
0,NCT01042717,Multiple Myeloma
1,NCT01042717,Non-Hodgkins Lymphoma
2,NCT04941040,Supratentorial Neoplasms
3,NCT05883540,Palliative Care
4,NCT05883540,Pain


In [37]:
conditions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8549 entries, 0 to 8548
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   study_id   8549 non-null   str  
 1   condition  8549 non-null   str  
dtypes: str(2)
memory usage: 133.7 KB


In [49]:
conditions_df = conditions_df.drop_duplicates()

## Extract Location Information
Geographic information was extracted to preserve study location data and enable future spatial analyses.

In [38]:
def extract_locations(trials):
    
    locations = []
    
    for trial in trials:
        
        protocol = trial["protocolSection"]
        
        study_id = protocol["identificationModule"].get("nctId")
        
        location_module = protocol.get(
            "contactsLocationsModule",
            {}
        )
        
        study_locations = location_module.get(
            "locations",
            []
        )
        
        for location in study_locations:
            
            locations.append({
                "study_id": study_id,
                "facility": location.get("facility"),
                "city": location.get("city"),
                "country": location.get("country")
            })
    
    return pd.DataFrame(locations)

In [39]:
locations_df = extract_locations(raw_trials)

In [40]:
locations_df.head()

,study_id,facility,city,country
0,NCT01042717,Mount Sinai School of Medicine,New York,United States
1,NCT04941040,Kasr El Aini Hospital,Cairo,Egypt
2,NCT05883540,"University Hospital Basel, Division of Clinica...",Basel,Switzerland
3,NCT05883540,"University Hospital Geneva, Palliative medicin...",Collonge-Bellerive,Switzerland
4,NCT05883540,"Spital Uster AG, Division of Internal Medicine",Uster,Switzerland


In [41]:
locations_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 57896 entries, 0 to 57895
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   study_id  57896 non-null  str  
 1   facility  53505 non-null  str  
 2   city      57892 non-null  str  
 3   country   57892 non-null  str  
dtypes: str(4)
memory usage: 1.8 MB


In [42]:
locations_df["country"].value_counts().head(10)

country
United States     27219
France             2641
Germany            2609
China              2418
Japan              2030
Spain              1941
Italy              1654
Canada             1539
United Kingdom     1499
Poland             1104
Name: count, dtype: int64

In [43]:
locations_df["country"].nunique()

121

In [44]:
locations_df.groupby("country")["study_id"].nunique().sort_values(ascending=False).head(10)

country
United States     2153
China              616
Germany            408
Canada             374
France             368
United Kingdom     332
Spain              324
Italy              271
South Korea        247
Australia          238
Name: study_id, dtype: int64

In [52]:
import os

os.makedirs("../data", exist_ok=True)

In [53]:
studies_df.to_csv("../data/studies.csv", index=False)
sponsors_df.to_csv("../data/sponsors.csv", index=False)
interventions_df.to_csv("../data/interventions.csv", index=False)
conditions_df.to_csv("../data/conditions.csv", index=False)
locations_df.to_csv("../data/locations.csv", index=False)

## Summary
The ClinicalTrials.gov API data was successfully transformed into five structured datasets: studies, sponsors, interventions, conditions, and locations. These datasets will be used in the following notebook to create a normalized relational SQLite database.